In [1]:
import jupyter_black
jupyter_black.load()

import re
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from ccl_science_data.common import get_arr, EntC, GenReader

loading  /home/borza/mega/hacking/apps/rankless/.env
OA_ROOT:  /home/borza/tmp/openalex-test-sets/micro-root
SNAP_ROOT:  /home/borza/tmp/openalex-test-sets/micro-snapshot


In [2]:
ns_rex = re.compile('impl NamespacedEntity for (.*) { const NS: & str = "(.*)"; }')

In [3]:
fgrs = ""
for grs in Path("../rankless_rs/src/gen/").iterdir():
    fgrs += "\n" + grs.read_text()

In [4]:
nsents = ns_rex.findall(fgrs)

In [5]:
numents = []
arrents = []
sdtmap = {}
for en, ns in nsents:
    numents += [
        [*e, ns]
        for e in re.findall(
            f'impl Entity for {en} {{ type T = u(\d+); const N: usize = \d+; const NAME: & str = "(.*)"; }}',
            fgrs,
        )
    ]
    for dtn, ename in re.findall(
        f'impl Entity for {en} {{ type T = Box<\[u(\d+)\]>; const N: usize = \d+; const NAME: & str = "(.*)"; }}',
        fgrs,
    ):
        arrents.append([dtn, ename, ns])
        sdtmap[ename] = re.findall(
            f"impl VariableSizeAttribute for {en} {{ type SizeType = u(\d+); type LocType = .*; }}",
            fgrs,
        )[0]

In [6]:
defs = ""
for dtn, en, ns in numents:
    defs += f"""
def larr_{en.replace('-', '_')}():
    return get_arr( "{ns}/{en}", {dtn} )    
"""

for dtn, en, ns in arrents:
    defs += f"""
def lvarr_{en.replace('-', '_')}():
    return get_arr( "{ns}/{en}/targets", {dtn} ), get_arr( "{ns}/{en}/sizes", {sdtmap[en]} )    
"""

In [7]:
exec(defs)

In [8]:
wyears = larr_work_years()
wcits = larr_work_citing_counts()

In [9]:
wtts, wtsis = lvarr_work_topics()

In [10]:
tsufs = larr_topic_subfields()

In [18]:
nsfs = tsufs.max() + 1

In [19]:
wsufs = np.zeros((wtsis.shape[0], nsfs), dtype=np.float16)

i = 0
for wi, wtsize in enumerate(tqdm(wtsis)):
    for j in range(wtsize):
        sufid = tsufs[wtts[i]]
        wsufs[wi, sufid] += 1 / wtsize
        i += 1

100%|████████████████████████████████████████████████████████████████| 215072/215072 [00:01<00:00, 195247.82it/s]


In [12]:
pd.Series(wsufs.sum(axis=1)).value_counts()

1.0    214697
0.0       375
Name: count, dtype: int64

In [91]:
discard_global_top = 0.005
discard_sf_top = 0.008
min_papers = 5
min_cites = 50

In [169]:
filt_arr = (wcits > 0) & (wcits < np.quantile(wcits, 1 - discard_global_top))

In [170]:
filt_arr.sum()

165474

In [171]:
dropsuf = []
for sfid in tqdm(range(nsfs)):
    sf_farr = wsufs[:, sfid] > 0.6
    if (sf_farr.sum() < min_papers) or (wcits[sf_farr].sum() < min_cites):
        dropsuf.append(sfid)
        filt_arr &= ~sf_farr
        continue
    filt_arr &= ~((wcits >= np.quantile(wcits[sf_farr], 1 - discard_sf_top)) & sf_farr)

100%|█████████████████████████████████████████████████████████████████████████| 253/253 [00:00<00:00, 511.62it/s]


In [172]:
filt_arr.sum()

164667

In [173]:
wsufs.shape

(215072, 253)

In [174]:
filt_wsufs = wsufs[:, ~np.isin(np.arange(nsfs), dropsuf)]
filt_wsufs = filt_wsufs / filt_wsufs.sum(axis=1).reshape(-1, 1)

/tmp/ipykernel_3208557/1335660219.py:2: RuntimeWarning: invalid value encountered in divide
  filt_wsufs = filt_wsufs / filt_wsufs.sum(axis=1).reshape(-1, 1)


In [175]:
filt_arr &= ~np.isnan(filt_wsufs).any(axis=1)

In [176]:
filt_arr.sum()

164253

In [178]:
act_year = wyears.max()

In [187]:
out_since = (act_year - wyears)[filt_arr]
f_wcits = wcits[filt_arr]
f_wsufs = filt_wsufs[filt_arr, :]

In [188]:
print(f_wsufs.shape, out_since.shape, f_wcits.shape)

(164253, 142) (164253,) (164253,)


In [ ]:
import numpy as np
import torch

# Suppose your arrays are:
# wsufs: (N, 142), f16
# out_since: (N,), int
# wcites: (N,), int

wsufs = torch.tensor(f_wsufs.astype(np.float32), dtype=torch.float32)
out_since = torch.tensor(out_since.astype(np.float32), dtype=torch.float32).view(-1, 1)
wcites = torch.tensor(f_wcits.astype(np.float32), dtype=torch.float32)
num_features = wsufs.shape[1]

# Parameters: alpha (coeff1) and b (logit for coeff2)
alpha = torch.randn(num_features, requires_grad=True)
b = torch.randn(num_features, requires_grad=True)

optimizer = torch.optim.Adam([alpha, b], lr=0.007)

for step in tqdm(range(2000)):
    optimizer.zero_grad()
    # coeff2 = sigmoid(b) to keep it in (0,1)
    beta = torch.sigmoid(b)

    # shape (N, num_features)
    numer = 1 - beta**out_since   # broadcasting
    denom = 1 - beta
    frac = numer / denom

    pred = (wsufs * (alpha * frac)).sum(dim=1)  # sum over features

    loss = torch.mean((pred - wcites)**2)

    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(step, loss.item())


In [191]:
alpha

tensor([1.0810, 0.8253, 0.8015, 0.6989, 0.3883, 1.3380, 0.8593, 0.7268, 0.4028,
        1.1141, 2.0432, 0.6923, 1.9745, 1.3888, 0.6076, 0.8974, 0.5725, 0.5912,
        0.4708, 0.4108, 0.5949, 0.5740, 0.9893, 0.7332, 0.3054, 0.4026, 0.4584,
        0.7256, 0.5534, 0.8238, 0.5916, 0.4913, 0.6328, 0.8653, 0.9176, 0.3557,
        0.8777, 0.6067, 0.5929, 0.4592, 0.8683, 0.7552, 0.8956, 0.6352, 0.6157,
        0.6525, 0.8479, 0.5697, 1.2155, 1.5085, 0.6648, 0.8928, 0.4586, 0.7164,
        0.7621, 0.5414, 1.1701, 0.7346, 0.4693, 0.4158, 0.9694, 0.9391, 1.1681,
        0.4623, 0.5799, 1.1672, 1.5638, 0.5714, 1.0450, 0.7602, 1.8037, 0.7591,
        0.6500, 0.4816, 0.4935, 0.9766, 0.9544, 0.4446, 1.7912, 1.6258, 0.4949,
        0.2766, 0.7233, 1.5522, 0.4295, 0.5403, 0.6449, 0.6938, 0.9092, 0.6752,
        1.2759, 0.7963, 0.6269, 0.4396, 1.6004, 1.2677, 0.8148, 0.4967, 0.6501,
        0.5077, 1.7625, 0.8168, 0.4368, 0.5215, 0.6309, 0.5146, 0.4855, 0.8782,
        0.8038, 0.4879, 1.7101, 0.5187, 

In [190]:
beta

tensor([0.7918, 0.8397, 0.8878, 0.7662, 0.9058, 0.6678, 0.8744, 0.8555, 0.9484,
        0.7867, 0.3611, 0.8976, 0.4882, 0.5130, 0.8733, 0.9016, 0.8929, 0.8700,
        0.9675, 0.9410, 0.8981, 0.9009, 0.8574, 0.9346, 0.9334, 0.9190, 0.9669,
        0.8928, 0.8272, 0.8588, 0.9258, 0.7809, 0.8129, 0.9273, 0.9065, 0.9275,
        0.8478, 0.8858, 0.8847, 0.9683, 0.7006, 0.8365, 0.5146, 0.9250, 0.9213,
        0.8580, 0.7689, 0.9149, 0.6225, 0.7722, 0.8433, 0.7370, 0.9447, 0.9345,
        0.7134, 0.8970, 0.4037, 0.6867, 0.9525, 0.9146, 0.7348, 0.8579, 0.7027,
        0.8328, 0.8944, 0.8338, 0.1330, 0.9105, 0.7060, 0.8656, 0.6829, 0.8872,
        0.8130, 0.9509, 0.8794, 0.7188, 0.8098, 0.9318, 0.4395, 0.3335, 0.9094,
        0.9346, 0.7943, 0.3814, 0.9710, 0.9147, 0.6156, 0.6881, 0.6063, 0.9092,
        0.7430, 0.6509, 0.8834, 0.7849, 0.2187, 0.6743, 0.6731, 0.8986, 0.9522,
        0.8991, 0.7863, 0.9313, 0.9915, 0.8671, 0.8503, 0.9779, 0.9354, 0.7880,
        0.9002, 0.9620, 0.4137, 0.8806, 

In [192]:
from ccl_science_data.common import GenReader, get_arr

In [193]:
gr = GenReader("..")

In [202]:
recs = []
j = 0
for i, name in enumerate(gr.get_names(EntC.SUBFIELDS)):
    if i in dropsuf:
        continue
    recs.append({"name": name, "decay": float(beta[j]), "volume": float(alpha[j])})
    j += 1

In [208]:
pdf = pd.DataFrame(recs).sort_values("volume", ascending=False)

In [212]:
pdf

,name,decay,volume
10,Genetics,0.361120,2.043191
12,Physiology,0.488196,1.974499
112,Sensory Systems,0.381604,1.876489
70,Condensed Matter Physics,0.682942,1.803682
78,Hepatology,0.439456,1.791183
...,...,...,...
134,Developmental and Educational Psychology,0.955085,0.312065
24,Education,0.933427,0.305435
140,Archeology,0.915938,0.284888
81,Gender Studies,0.934586,0.276635


In [211]:
pdf.set_index("name").corr()

,decay,volume
decay,1.000000,-0.790819
volume,-0.790819,1.000000
